# 🛣️ ROADGUARD AI — HUẤN LUYỆN MODEL NHẬN DIỆN VẾT NỨT ĐƯỜNG (YOLOv11)
### Huấn luyện trên Google Colab GPU T4 | Tự động tải Dataset & Xuất file trọng số `crack_best.pt`
---
> **Mục tiêu:** Nhận diện các dạng hư hỏng mặt đường: Nứt dọc (Longitudinal), Nứt ngang (Transverse), Nứt chân chim (Alligator), và Ổ gà (Pothole).
> **Thiết bị chạy:** GPU Tesla T4 (Miễn phí trên Google Colab).

## 📌 Bước 1: Kiểm Tra GPU T4 & Cài Đặt Thư Viện

In [ ]:
# 1. Kiểm tra phần cứng GPU
!nvidia-smi

import torch
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn T4 GPU")
print("=" * 60)

# 2. Cài đặt Ultralytics và Roboflow
!pip install -q ultralytics roboflow

## 📥 Bước 2: Tải Dữ Liệu Vết Nứt Mặt Đường (Road Crack Dataset)

In [ ]:
import os, glob, zipfile

# Tải dataset vết nứt đường chuẩn hóa YOLO từ Roboflow Universe
# Nếu bạn có API Key Roboflow riêng, có thể thay thế vào bên dưới.
# Hoặc tải tập dữ liệu public Road Crack đã đóng gói sẵn:

print("📥 Đang tải tập dữ liệu Vết nứt đường (Road Crack & Pothole Dataset)...")
!curl -L -o crack_dataset.zip "https://universe.roboflow.com/ds/t56nF4p95k?key=v7O8273x0P" || echo "Curl fallback"

# Tự động kiểm tra và giải nén
dataset_dir = "/content/dataset"
os.makedirs(dataset_dir, exist_ok=True)

zip_files = glob.glob("*.zip") + glob.glob("/content/*.zip")
print(f"Tìm thấy các file zip: {zip_files}")

if zip_files:
    chosen_zip = zip_files[0]
    print(f"Đang giải nén {chosen_zip} vào {dataset_dir}...")
    with zipfile.ZipFile(chosen_zip, 'r') as z:
        z.extractall(dataset_dir)
    print("✅ Giải nén dataset thành công!")

# Quét tìm file data.yaml
yaml_candidates = glob.glob(f"{dataset_dir}/**/data.yaml", recursive=True) + glob.glob("**/data.yaml", recursive=True)
if yaml_candidates:
    yaml_path = os.path.abspath(yaml_candidates[0])
    print(f"✅ Đã định vị cấu hình Dataset tại: {yaml_path}")
    with open(yaml_path, 'r') as f:
        print("--- NỘI DUNG DATA.YAML ---")
        print(f.read())
else:
    print("⚠️ Chưa thấy data.yaml. Bạn có thể kéo thả file zip dataset của bạn vào Colab.")

## 🧠 Bước 3: Khởi Tạo & Huấn Luyện Mô Hình YOLOv11

In [ ]:
from ultralytics import YOLO

# 1. Khởi tạo mô hình YOLOv11 Nano (tối ưu tốc độ cao và suy luận thời gian thực cho Drone)
model = YOLO('yolo11n.pt')

# 2. Bắt đầu huấn luyện 50 Epochs
print("🚀 BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN YOLOv11...")
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # Sử dụng GPU T4
    workers=4,
    save=True,
    save_period=10,
    plots=True,
    name='roadguard_crack_v1'
)

print("🎉 HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")

## 📊 Bước 4: Đánh Giá Kết Quả Huấn Luyện (Loss & mAP50 Metrics)

In [ ]:
import glob
from IPython.display import Image, display

run_dir = "runs/detect/roadguard_crack_v1"
print(f"Thư mục kết quả: {run_dir}")

# Hiển thị biểu đồ hội tụ (Training curves & Losses)
results_img = f"{run_dir}/results.png"
if os.path.exists(results_img):
    print("📈 Biểu đồ hàm mất mát (Loss) & Độ chính xác mAP:")
    display(Image(results_img))

# Hiển thị ma trận nhầm lẫn (Confusion Matrix)
cm_img = f"{run_dir}/confusion_matrix.png"
if os.path.exists(cm_img):
    print("🎯 Ma trận nhầm lẫn (Confusion Matrix):")
    display(Image(cm_img))

# Hiển thị mẫu ảnh dự đoán trên tập kiểm thử (Validation Batch Predictions)
val_imgs = glob.glob(f"{run_dir}/val_batch*_pred.jpg")
if val_imgs:
    print("🔍 Ảnh dự đoán thử nghiệm:")
    display(Image(val_imgs[0]))

## 💾 Bước 5: Tải File Trọng Số `best.pt` Về Máy Tính

In [ ]:
import shutil
from google.colab import files

best_weight_src = f"{run_dir}/weights/best.pt"
target_weight = "crack_best.pt"

if os.path.exists(best_weight_src):
    shutil.copy(best_weight_src, target_weight)
    print(f"✅ Đã chuẩn bị file trọng số: {target_weight}")
    print("⬇️ Đang kích hoạt tải file về máy tính của bạn...")
    files.download(target_weight)
    print("👉 Sau khi tải về, hãy chép file vào thư mục: f:\\train AI đồ án\\weights\\trained\\crack_best.pt")
else:
    print(f"❌ Không tìm thấy file {best_weight_src}")